In [1]:
import os

In [2]:
%pwd

'c:\\Users\\KIIT0001\\Desktop\\Chest-cancer-classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\KIIT0001\\Desktop\\Chest-cancer-classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [6]:
from Chest_Cancer_classifier.constants import *
from Chest_Cancer_classifier.utils.common import read_yaml , create_directories
import tensorflow as tf

[2026-04-26 19:17:18,889: INFO: utils: NumExpr defaulting to 12 threads.]


In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

        

    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "Chest_CT_scan_DATA")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [8]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [11]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        # Load architecture/weights only; we will attach a fresh optimizer below.
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path,
            compile=False
        )

    def _compile_model(self):
        self.model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self):
        self._compile_model()
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [13]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e

[2026-04-26 19:22:11,864: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-26 19:22:11,877: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-26 19:22:11,878: INFO: common: created directory at: artifacts]
[2026-04-26 19:22:11,880: INFO: common: created directory at: artifacts\training]
Found 68 images belonging to 2 classes.
Found 275 images belonging to 2 classes.
Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 37s 2s/step - accuracy: 0.5251 - loss: 14.2570 - val_accuracy: 0.6094 - val_loss: 19.3590
Epoch 2/10
 1/17 ━━━━━━━━━━━━━━━━━━━━ 32s 2s/step - accuracy: 0.5625 - loss: 22.9483

c:\Users\KIIT0001\Desktop\Chest-cancer-classification\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 550ms/step - accuracy: 0.5625 - loss: 22.9483 - val_accuracy: 0.7812 - val_loss: 0.3825
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 44s 3s/step - accuracy: 0.5598 - loss: 14.0790 - val_accuracy: 0.6094 - val_loss: 11.8562
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 12s 583ms/step - accuracy: 0.6250 - loss: 13.9191 - val_accuracy: 0.7969 - val_loss: 0.6024
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 44s 3s/step - accuracy: 0.5676 - loss: 11.9758 - val_accuracy: 0.8750 - val_loss: 0.7882
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 550ms/step - accuracy: 0.6875 - loss: 1.0013 - val_accuracy: 0.4062 - val_loss: 6.3892
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 45s 3s/step - accuracy: 0.6139 - loss: 7.2094 - val_accuracy: 0.5938 - val_loss: 4.3121
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 11s 549ms/step - accuracy: 0.7500 - loss: 5.6448 - val_accuracy: 0.8750 - val_loss: 0.1543
Epoch 9/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 44s 3s/step - accuracy: 0.7375 - loss: 4.4318 - val_accuracy: 0.8750 - val